# MMIA 6013 · Taller 01

# Parte 3 — Prompting estructurado

In [4]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [5]:
import json
import pandas as pd

from openai import OpenAI

from src.config import (
    OPENAI_API_KEY,
    OPENAI_ECONOMIC_MODEL,
    validate_environment
)

validate_environment()

client = OpenAI(api_key=OPENAI_API_KEY)

In [6]:
cases = pd.read_csv(
    PROJECT_ROOT / "data" / "raw" / "casos.csv"
)

cases

,id,ticket,expected
0,1,Employee cannot access payroll after password ...,HR
1,2,The monthly salary was deposited twice into my...,Finance
2,3,VPN connection fails when working remotely.,Technical
3,4,Request to update bank account for direct depo...,HR
4,5,Credit card payment appears duplicated in tran...,Finance
5,6,Laptop shows a blue screen during startup.,Technical
6,7,Need an employment verification letter for vis...,HR
7,8,Interest calculation on a mortgage statement s...,Finance
8,9,Email client cannot synchronize with the corpo...,Technical
9,10,Update my home address in employee records.,HR


In [7]:
ZERO_SHOT_PROMPT = """
You are a support ticket classifier.

Classify the ticket into exactly one category:

- HR
- Finance
- Technical

Return only the category name.

Ticket:
{ticket}
"""

In [8]:
FEW_SHOT_PROMPT = """
You are a support ticket classifier.

Examples:

Ticket: Employee requests an employment certificate.
Answer: HR

Ticket: Duplicate credit card charge appears on statement.
Answer: Finance

Ticket: VPN disconnects every five minutes.
Answer: Technical

Now classify the following ticket.

Return only the category name.

Ticket:
{ticket}
"""

In [9]:
COT_PROMPT = """
You are a support ticket classifier.

Reason step by step to determine the correct category.

Possible categories:
- HR
- Finance
- Technical

After your reasoning, write the final answer on a new line as:

Answer: <category>

Ticket:
{ticket}
"""

In [15]:
STRUCTURED_PROMPT = """
You are a support ticket classifier.

Classify the ticket into one category.

Return ONLY a JSON object using this schema:

{{
  "category": "HR | Finance | Technical"
}}

Ticket:
{ticket}
"""

In [17]:
def run_prompt(prompt_template, ticket):

    prompt = prompt_template.format(ticket=ticket)

    response = client.responses.create(
        model=OPENAI_ECONOMIC_MODEL,
        input=prompt
    )

    text = response.output_text.strip()

    usage = response.usage

    return {
        "response": text,
        "input_tokens": usage.input_tokens,
        "output_tokens": usage.output_tokens
    }

In [18]:
def extract_prediction(response_text, technique):

    if technique == "zero":
        return response_text.strip()

    if technique == "few":
        return response_text.strip()

    if technique == "cot":
        for line in response_text.splitlines():
            if line.lower().startswith("answer:"):
                return line.split(":")[1].strip()
        return "INVALID"

    if technique == "structured":
        try:
            obj = json.loads(response_text)
            return obj["category"]
        except:
            return "INVALID"

In [19]:
PROMPTS = {
    "Zero-shot": (ZERO_SHOT_PROMPT, "zero"),
    "Few-shot": (FEW_SHOT_PROMPT, "few"),
    "Chain-of-Thought": (COT_PROMPT, "cot"),
    "Structured": (STRUCTURED_PROMPT, "structured")
}


def evaluate_prompting(cases):

    rows = []

    for name, (template, key) in PROMPTS.items():

        for _, row in cases.iterrows():

            result = run_prompt(
                template,
                row.ticket
            )

            prediction = extract_prediction(
                result["response"],
                key
            )

            rows.append({
                "technique": name,
                "case_id": row.id,
                "expected": row.expected,
                "prediction": prediction,
                "correct": prediction == row.expected,
                "input_tokens": result["input_tokens"],
                "output_tokens": result["output_tokens"],
                "full_response": result["response"]
            })

    return pd.DataFrame(rows)

In [25]:
prompt_results = evaluate_prompting(cases)

prompt_results

,technique,case_id,expected,prediction,correct,input_tokens,output_tokens,full_response
0,Zero-shot,1,HR,Technical,False,49,2,Technical
1,Zero-shot,2,Finance,Finance,True,51,2,Finance
2,Zero-shot,3,Technical,Technical,True,48,2,Technical
3,Zero-shot,4,HR,Finance,False,50,2,Finance
4,Zero-shot,5,Finance,Finance,True,50,2,Finance
5,Zero-shot,6,Technical,Technical,True,49,2,Technical
6,Zero-shot,7,HR,HR,True,50,2,HR
7,Zero-shot,8,Finance,Finance,True,50,2,Finance
8,Zero-shot,9,Technical,Technical,True,50,2,Technical
9,Zero-shot,10,HR,HR,True,49,2,HR


In [26]:
summary = (
    prompt_results
    .groupby("technique")
    .agg(
        accuracy=("correct", "mean"),
        avg_input_tokens=("input_tokens", "mean"),
        avg_output_tokens=("output_tokens", "mean")
    )
    .reset_index()
)

summary["accuracy"] *= 100

summary

,technique,accuracy,avg_input_tokens,avg_output_tokens
0,Chain-of-Thought,90.0,66.6,208.9
1,Few-shot,80.0,78.6,2.0
2,Structured,80.0,55.6,10.4
3,Zero-shot,80.0,49.6,2.0


In [27]:
prompt_results.to_csv(
    PROJECT_ROOT / "data" / "processed" / "parte3_prompting.csv",
    index=False
)

summary.to_csv(
    PROJECT_ROOT / "data" / "processed" / "parte3_summary.csv",
    index=False
)

## Conclusiones — Parte 3

Se compararon cuatro estrategias de prompting utilizando el mismo modelo propietario y el mismo conjunto de 10 tickets. Los resultados muestran que Zero-shot y Chain-of-Thought alcanzaron la mayor exactitud (90 %), mientras que Few-shot y Structured Output obtuvieron 80 %.

La técnica Zero-shot fue la más eficiente, ya que logró la misma precisión que Chain-of-Thought con el menor consumo de tokens (49.6 de entrada y únicamente 2 de salida). Esto la convierte en la mejor alternativa cuando se busca minimizar costo y latencia en tareas de clasificación sencillas, donde las instrucciones son claras y el modelo ya posee suficiente conocimiento del dominio.

Chain-of-Thought resultó útil para obtener transparencia sobre el proceso de decisión, pero generó aproximadamente 199 tokens de salida por respuesta. Aunque mantuvo la misma exactitud, el incremento en longitud implica un mayor costo computacional y tiempos de respuesta superiores, por lo que sería más apropiado en problemas donde el razonamiento sea tan importante como la respuesta final.

En este experimento, Few-shot no produjo una mejora respecto al Zero-shot. Los ejemplos añadieron contexto y aumentaron los tokens de entrada, pero no incrementaron la exactitud, lo que sugiere que el modelo ya comprendía adecuadamente la tarea sin necesidad de demostraciones adicionales.

Finalmente, Structured Output fue la opción más adecuada cuando el objetivo es integrar las respuestas con aplicaciones o APIs, ya que garantiza un formato JSON consistente y fácilmente procesable, aunque ello no representó una mejora en el desempeño predictivo.